In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (calibration still runs):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, json, os
from pathlib import Path
adapter_dirs = {}
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    v = variant.split("_")[0]
    cands = sorted(glob.glob(f"/kaggle/input/**/{v}/adapter", recursive=True))
    cands += sorted(glob.glob("/kaggle/input/arc-tier-b-adapters/{v}/adapter"))
    hits = [c for c in cands if (Path(c) / "adapter_config.json").exists()]
    if hits:
        adapter_dirs[variant] = hits[0]
        print(f"adapter [{variant}] ->{hits[0]}")
    else:
        print(f"!! NO adapter output for [{variant}]")
if not adapter_dirs:
    print("INPUT DIRS:", os.listdir("/kaggle/input"))
    for root in (Path("/kaggle/input")).rglob("adapter_config.json"):
        print("found adapter_config:", root)
    print("DATASETS DIR:", os.listdir("/kaggle/input/datasets") if os.path.isdir("/kaggle/input/datasets") else "n/a")
    raise SystemExit("No adapters found - is dataset niyuvo/arc-tier-b-adapters attached?")
open("/kaggle/working/adapters.json", "w").write(json.dumps(adapter_dirs, indent=2))


In [ ]:
import json, os, subprocess, sys
adapter_dirs = json.load(open("/kaggle/working/adapters.json"))
CONFIGS = [["default", {"bias": 0.6, "halt_threshold": 0.45, "min_gain": 0.02}], ["early1", {"bias": 0.5, "halt_threshold": 0.4, "min_gain": 0.03}], ["early2", {"bias": 0.45, "halt_threshold": 0.4, "min_gain": 0.04}], ["early3", {"bias": 0.45, "k": 10, "halt_threshold": 0.35, "min_gain": 0.05}], ["early4", {"bias": 0.4, "k": 10, "halt_threshold": 0.35, "min_gain": 0.06}], ["gain8", {"bias": 0.5, "k": 12, "halt_threshold": 0.45, "min_gain": 0.08}], ["settle", {"bias": 0.55, "k": 12, "halt_threshold": 0.5, "min_gain": 0.02}]]
tasks = "arc_easy,piqa"
limits = "arc_easy=30,piqa=30"
os.makedirs("/kaggle/working/calib", exist_ok=True)
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    for cname, ckw in CONFIGS:
        out = f"/kaggle/working/calib/calib-{variant}-{cname}.csv"
        cmd = [sys.executable, "scripts/run_benchmarks.py",
               "--base", "/kaggle/working/jetmoe-8b",
               "--model", variant,
               "--adapters", f"{variant}={adapter_dirs[variant]}",
               "--budgeted", "--max_loops", "4",
               "--tasks", tasks, "--limits", limits,
               "--controller", json.dumps(ckw),
               "--out", out]
        print(">>>", variant, cname, json.dumps(ckw))
        subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)


In [ ]:
import csv, glob, math, os, shutil, json
os.makedirs("/kaggle/output/calib", exist_ok=True)
csvs = sorted(glob.glob("/kaggle/working/calib/calib-*.csv"))
for c in csvs:
    shutil.copy(c, f"/kaggle/output/calib/{{os.path.basename(c)}}")
print(f"{{len(csvs)}} calibration CSVs saved")
print()
mcq = ["arc_easy", "piqa"]
for key in ["model_adaptive", "block_adaptive", "layer_adaptive"]:
    print("\n=== ", key, " ===")
    print(f"{'config':10s} {'task':9s} acc   loops  decide  nan   halts_at")
    for c in sorted(glob.glob(f"/kaggle/working/calib/calib-{key}-*.csv")):
        cname = os.path.basename(c).replace(f"calib-{key}-", "").replace(".csv", "")
        rows = list(csv.DictReader(open(c)))
        for r in rows:
            acc = float(r["acc"]) * 100
            print(f"{cname:10s} {r['task']:9s} {acc:5.1f}%  "
                  f"{r['avg_loops_per_item']:>5} {r['avg_decide_mean_p']:>6} "
                  f"{r['nan_halts']:>4} {r['halt_hist']}")
